# 03 — Train Retriever (M4): fine-tune bi-encoder với hard negatives

Input: corpus chunk từ `data/samples/corpus/` (chunk bằng `rag/chunker.py`) + cặp (câu
trong mục E-HSMT đã duyệt, chunk nguồn tương ứng) làm positive pair.
Hard negative: lấy top-k chunk có BM25 score cao nhưng KHÔNG phải chunk đúng.
Output: checkpoint tại `models/retriever_bi_encoder/`.
Metric: Recall@5, MRR@10, nDCG@10, so sánh với baseline BM25 (`rag/bm25.py`).

In [ ]:
!pip install -q sentence-transformers faiss-cpu

In [ ]:
import sys
sys.path.insert(0, '/content/autotender-vn/src')
from autotender.rag.chunker import chunk_corpus_dir
from autotender.rag.bm25 import build_bm25_index

chunks = chunk_corpus_dir('/content/autotender-vn/data/samples/corpus')
texts = [c.text for c in chunks]
print(len(chunks), 'chunks')

## Cặp huấn luyện

**LƯU Ý (giới hạn nghiên cứu):** đồ án chưa có tập câu hỏi/câu-truy-vấn thật gán nhãn
với chunk đúng (cần gán tay hoặc thu thập từ log chỉnh sửa HITL — xem `hitl/feedback.py`).
Ở đây minh hoạ bằng cách coi `query` của mỗi `SECTION_DEFINITIONS` (`models/generator.py`)
là positive query cho chunk có điểm BM25 cao nhất, để pipeline chạy được end-to-end.
Khi có dữ liệu phản hồi HITL thật, thay bước này bằng cặp (câu đã người dùng phê duyệt, chunk trích dẫn).

In [ ]:
from autotender.models.generator import SECTION_DEFINITIONS

bm25 = build_bm25_index(texts)
train_examples = []
for section_id, defn in SECTION_DEFINITIONS.items():
    query = defn['query']
    ranked = bm25.search(query, top_k=10)
    if not ranked:
        continue
    pos_idx = ranked[0][0]
    hard_neg_idx = [i for i, _ in ranked[1:6]]
    train_examples.append((query, texts[pos_idx], [texts[i] for i in hard_neg_idx]))
print(len(train_examples), 'training triples (query, positive, hard_negatives)')

In [ ]:
from sentence_transformers import InputExample, SentenceTransformer, losses
from torch.utils.data import DataLoader

MODEL_NAME = 'bkai-foundation-models/vietnamese-bi-encoder'
model = SentenceTransformer(MODEL_NAME)

examples = [InputExample(texts=[q, pos]) for q, pos, _negs in train_examples]
loader = DataLoader(examples, shuffle=True, batch_size=8)
loss = losses.MultipleNegativesRankingLoss(model)
model.fit(train_objectives=[(loader, loss)], epochs=5, warmup_steps=10, show_progress_bar=True)
model.save('/content/models/retriever_bi_encoder')
print('Checkpoint saved — tải về models/retriever_bi_encoder/ trong repo local.')

## Đánh giá: Recall@5, MRR@10, nDCG@10 vs BM25 (ablation Mục 10)

In [ ]:
import numpy as np

def evaluate_retriever(scorer_fn, top_k=5):
    recalls, rr = [], []
    for query, positive, _negs in train_examples:
        ranked_texts = scorer_fn(query, top_k=10)
        hit_rank = next((i for i, t in enumerate(ranked_texts) if t == positive), None)
        recalls.append(1.0 if hit_rank is not None and hit_rank < top_k else 0.0)
        rr.append(1.0 / (hit_rank + 1) if hit_rank is not None else 0.0)
    return {'recall@%d' % top_k: float(np.mean(recalls)), 'mrr@10': float(np.mean(rr))}

def bm25_scorer(query, top_k):
    ranked = bm25.search(query, top_k=top_k)
    return [texts[i] for i, _ in ranked]

def bi_encoder_scorer(query, top_k):
    q_emb = model.encode(query)
    t_emb = model.encode(texts)
    sims = t_emb @ q_emb
    order = np.argsort(-sims)[:top_k]
    return [texts[i] for i in order]

print('BM25 baseline:', evaluate_retriever(bm25_scorer))
print('Fine-tuned bi-encoder:', evaluate_retriever(bi_encoder_scorer))